# Neural Network From Scratch


## A. Neural network: binary classification

A fully-connected neural network with one hidden layer, built entirely in numpy on the Breast Cancer Wisconsin dataset (569 samples, 30 real-valued features, malignant vs benign).

Every component is written by hand: the network architecture, random parameter initialization, forward propagation with a `tanh` hidden layer, the cross-entropy cost, backpropagation, and the gradient descent update. scikit-learn supplies only the data and the train/test split.

This picks up where the logistic regression notebook in this folder left off. There the model was a single neuron; here a hidden layer is added, which is what makes the network able to learn a non-linear boundary.


In [1]:
# Package imports
import numpy as np
import sklearn
import sklearn.linear_model
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn import preprocessing
import matplotlib.pyplot as plt

%matplotlib inline

np.random.seed(1)

## 1. Loading the dataset and preprocessing


In [2]:
X, y = load_breast_cancer(return_X_y=True)
train_data, test_data, train_labels, test_labels = train_test_split(X, y, test_size =0.2, random_state=0)

In [3]:
scaler = preprocessing.StandardScaler().fit(train_data)
train_data = scaler.transform(train_data)
test_data = scaler.transform(test_data)

In [4]:
trainx = train_data.T
trainy = train_labels.reshape(-1,1).T

testx = test_data.T
testy =test_labels.reshape(-1,1).T

In [5]:
trainx.shape, trainy.shape, testx.shape, testy.shape

((30, 455), (1, 455), (30, 114), (1, 114))

In [6]:
X=trainx
Y=trainy

In [7]:
shape_X = X.shape
shape_Y = Y.shape
m = X.shape[1]  # training set size

print ('No. of training samples: ' + str(m))
print ('Number of features per sample: ' + str(shape_X[0]))

No. of training samples: 455
Number of features per sample: 30


## 2 - Neural Network model

A neural network with a single hidden layer.

**Mathematically**:

For one example $x^{(i)}$:
$$z^{[1] (i)} =  W^{[1]} x^{(i)} + b^{[1]}\tag{1}$$ 
$$a^{[1] (i)} = \tanh(z^{[1] (i)})\tag{2}$$
$$z^{[2] (i)} = W^{[2]} a^{[1] (i)} + b^{[2]}\tag{3}$$
$$\hat{y}^{(i)} = a^{[2] (i)} = \sigma(z^{ [2] (i)})\tag{4}$$
$$y^{(i)}_{prediction} = \begin{cases} 1 & \mbox{if } a^{[2](i)} > 0.5 \\ 0 & \mbox{otherwise } \end{cases}\tag{5}$$

Given the predictions on all the examples, the cost $J$ is: 
$$J = - \frac{1}{m} \sum\limits_{i = 0}^{m} \large\left(\small y^{(i)}\log\left(a^{[2] (i)}\right) + (1-y^{(i)})\log\left(1- a^{[2] (i)}\right)  \large  \right) \small \tag{6}$$

**Important**: Building the NN will involve the following:
    1. Specify the network structure in terms of the number of input units, number of neurons in the hidden units, ...
    2. Initialize the parameters of the model
    3. Loop a number of iterations:
        - Forward propagation
        - Compute loss and the overall cost
        - Backward propagation
        - Update the parameters (gradient descent)

In order to make the code modular, we can implement each of the step as a function and them combine them together to build the overall model.


### 2.1 - Specifying the network structure

Three sizes define the network:

- `n_x`: input layer size, one unit per feature
- `n_h`: number of neurons in the hidden layer
- `n_y`: output layer size


In [9]:
def model_architecture(X, Y):
    """
    Arguments:
    X -- input dataset of shape (input size, number of examples)
    Y -- labels of shape (output size, number of examples)
    
    Returns:
    n_x -- the size of the input layer
    n_h -- the size of the hidden layer
    n_y -- the size of the output layer
    """
    n_x = X.shape[0] # size of input layer
    n_h = 10
    n_y = Y.shape[0] # size of output layer
    return (n_x, n_h, n_y)

### 2.2 - Initializing the parameters

The weight matrices are initialized to small random values with `np.random.randn(a, b) * 0.01`, and the bias vectors to zeros with `np.zeros((a, b))`.

The random initialization is essential and is the one real difference from logistic regression. If every weight started at zero, all hidden units would compute exactly the same thing, receive exactly the same gradient, and remain identical for the entire training run, leaving a network no more expressive than a single neuron. Breaking that symmetry is the whole reason for the randomness. The biases can safely stay at zero, since the weights already break the symmetry.

The scale matters too: multiplying by 0.01 keeps the initial values small, which keeps `tanh` in its steep central region where gradients are large, rather than saturated at its flat tails.


In [10]:
def initialize_parameters(n_x, n_h, n_y):
    """
    Argument:
    n_x -- size of the input layer
    n_h -- size of the hidden layer
    n_y -- size of the output layer
    
    Returns:
    params -- python dictionary containing your parameters:
                    W1 -- weight matrix of shape (n_h, n_x)
                    b1 -- bias vector of shape (n_h, 1)
                    W2 -- weight matrix of shape (n_y, n_h)
                    b2 -- bias vector of shape (n_y, 1)
    """
    
    np.random.seed(2)

    
    W1 = np.random.randn(n_h, n_x) * 0.01
    b1 = np.zeros((n_h, 1))
    W2 = np.random.randn(n_y, n_h) * 0.01
    b2 = np.zeros((n_y, 1))
    
    assert (W1.shape == (n_h, n_x))
    assert (b1.shape == (n_h, 1))
    assert (W2.shape == (n_y, n_h))
    assert (b2.shape == (n_y, 1))
    
    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2}
    
    return parameters

### 2.3 - The loop

With the structure and parameters set, the training loop needs four pieces: the sigmoid, forward propagation, the cost, and backpropagation.

Forward propagation retrieves each parameter from the `parameters` dictionary, then computes $Z^{[1]}, A^{[1]}, Z^{[2]}$ and $A^{[2]}$, caching the intermediate values that backpropagation will need. The hidden layer uses `np.tanh()` and the output layer a sigmoid, since this is binary classification.


In [11]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [12]:
def forward_propagation(X, parameters):
    """
    Argument:
    X -- input data of size (n_x, m)
    parameters -- python dictionary containing your parameters (output of initialization function)
    
    Returns:
    A2 -- The sigmoid output of the second activation
    cache -- a dictionary containing "Z1", "A1", "Z2" and "A2"
    """
    # Retrieve each parameter from the dictionary "parameters"
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    
    # Implement Forward Propagation to calculate A2 (probabilities)
    Z1 = np.dot(W1, X) + b1
    A1 = np.tanh(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)
    
    assert(A2.shape == (1, X.shape[1]))
    
    cache = {"Z1": Z1,
             "A1": A1,
             "Z2": Z2,
             "A2": A2}
    
    return A2, cache

With $A^{[2]}$ computed, the cross-entropy cost follows:

$$J = - \frac{1}{m} \sum\limits_{i = 1}^{m} \large{(} \small y^{(i)}\log\left(a^{[2] (i)}\right) + (1-y^{(i)})\log\left(1- a^{[2] (i)}\right) \large{)} \small\tag{7}$$


In [13]:
def compute_cost(A2, Y):
    """
    Arguments:
    A2 -- The sigmoid output of the second activation, of shape (1, number of examples)
    Y -- "true" labels vector of shape (1, number of examples)
       
    Returns:
    cost -- cross-entropy cost given equation (7)
    
    """
    
    m = Y.shape[1] # number of example

    # Compute the cross-entropy cost
    logprobs = np.multiply(np.log(A2), Y) + np.multiply(np.log(1 - A2), 1 - Y)
    cost = - np.sum(logprobs) / m
    
    cost = float(np.squeeze(cost))

    assert(isinstance(cost, float))
    
    return cost

Using the values cached during forward propagation, backpropagation walks the gradients back through the network. The six vectorised equations:

$$dZ^{[2]} = A^{[2]} - Y \tag{8}$$
$$dW^{[2]} = \frac{1}{m} dZ^{[2]}A^{[1]{T}} \tag{9}$$
$$db^{[2]} = \frac{1}{m} np.sum(dZ^{[2]}, axis = 1, keepdims = True)\tag{10}$$
$$dZ^{[1]} = W^{[2]T}dZ^{[2]}*g^{[1]'}(Z^{[1]}) \tag{11}$$
$$dW^{[1]} = \frac{1}{m} dZ^{[1]}X^{{T}} \tag{12}$$
$$db^{[1]} = \frac{1}{m} np.sum(dZ^{[1]}, axis = 1, keepdims = True)\tag{13}$$

Equation (11) is where the chain rule does its work: the error at the output is projected back through $W^{[2]}$ and then scaled by the derivative of the hidden activation, $g^{[1]'}(Z^{[1]}) = 1 - \tanh^2(Z^{[1]})$.


In [14]:
def backprop(parameters, cache, X, Y):
    """
    Arguments:
    parameters -- python dictionary containing our parameters 
    cache -- a dictionary containing "Z1", "A1", "Z2" and "A2".
    X -- input data
    Y -- "true" labels
    
    Returns:
    grads -- python dictionary containing your gradients with respect to different parameters
    """
    m = X.shape[1]
    
    # First, retrieve W1 and W2 from the dictionary "parameters".
    W1 = parameters["W1"]
    W2 = parameters["W2"]
        
    # Retrieve also A1 and A2 from dictionary "cache".
    A1 = cache["A1"]
    A2 = cache["A2"]
    
    # Backward propagation: calculate dW1, db1, dW2, db2. 
    dZ2 = A2 - Y
    dW2 = (1/m) * np.dot(dZ2, A1.T)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = np.dot(W2.T, dZ2) * (1 - np.power(A1, 2))
    dW1 = (1/m) * np.dot(dZ1, X.T)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)
    
    grads = {"dW1": dW1,
             "db1": db1,
             "dW2": dW2,
             "db2": db2}
    
    return grads

The parameters are then updated by gradient descent, using `(dW1, db1, dW2, db2)` to update `(W1, b1, W2, b2)`.

**Gradient descent rule**: $\theta = \theta - \alpha \frac{\partial J }{ \partial \theta }$, where $\alpha$ is the learning rate and $\theta$ represents a parameter.


In [15]:
def update(parameters, grads, learning_rate = 0.01):
    """
    Arguments:
    parameters -- python dictionary containing your parameters 
    grads -- python dictionary containing your gradients 
    learning_rate -- The learning rate
    
    Returns:
    parameters -- python dictionary containing your updated parameters 
    """
    # Retrieve each parameter from the dictionary "parameters"
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    
    # Retrieve each gradient from the dictionary "grads"
    dW1 = grads["dW1"]
    db1 = grads["db1"]
    dW2 = grads["dW2"]
    db2 = grads["db2"]
    
    # Update rule for each parameter
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1
    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2
    
    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2}
    
    return parameters

### 2.4 - Integrating everything into NeuralNetwork()

All the pieces above are assembled into a single `NeuralNetwork()` function, which calls them in order: define the architecture, initialize the parameters, then loop over forward propagation, cost, backpropagation and update.


In [17]:
def NeuralNetwork(X, Y, n_h, num_iterations = 10000, learning_rate = 0.01, print_cost=False):
    """
    Arguments:
    X -- dataset
    Y -- labels 
    n_h -- size of the hidden layer
    num_iterations -- Number of iterations in gradient descent loop
    learning_rate -- The learning rate
    print_cost -- if True, print the cost every 1000 iterations
    
    Returns:
    parameters -- parameters learnt by the model. They can then be used to make predictions.
    """
    
    np.random.seed(3)
    n_x = model_architecture(X, Y)[0]
    n_y = model_architecture(X, Y)[2]
    
    # Initialize parameters
    parameters = initialize_parameters(n_x, n_h, n_y)
    
    # Loop (gradient descent)

    for i in range(0, num_iterations):
         
        # Forward propagation. Inputs: "X, parameters". Outputs: "A2, cache".
        A2, cache = forward_propagation(X, parameters)
        
        # Cost function. Inputs: "A2, Y, parameters". Outputs: "cost".
        cost = compute_cost(A2, Y)
 
        # Backpropagation. Inputs: "parameters, cache, X, Y". Outputs: "grads".
        grads = backprop(parameters, cache, X, Y)
 
        # Gradient descent parameter update. Inputs: "parameters, grads". Outputs: "parameters".
        parameters =  update(parameters, grads, learning_rate)
        
        
        # Print the cost every 100 iterations
        if print_cost and i % 100 == 0:
            print ("Cost after iteration %i: %f" %(i, cost))

    return parameters

### 2.5 - Predictions

Prediction runs forward propagation and thresholds the output activation:

$$predictions = \begin{cases}
      1 & \text{if}\ activation > 0.5 \\
      0 & \text{otherwise}
    \end{cases}$$


In [20]:
def predict(parameters, X):
    """
    Arguments:
    parameters -- python dictionary containing your parameters 
    X -- input data 
    
    Returns
    predictions -- vector of predictions of our model
    """
    
    # Computes probabilities using forward propagation, and classifies to 0/1 using 0.5 as the threshold.
    A2, cache = forward_propagation(X, parameters)
    predictions = (A2 > 0.5) # it creates a boolean array -> True/False which acts as 1/0
    
    return predictions

## 3. Model execution

Training the network with a single hidden layer of $n_h$ units.


In [21]:
# Build a model with a n_h-dimensional hidden layer
parameters = NeuralNetwork(X, Y, n_h = 10, num_iterations = 10000, print_cost=True)

Cost after iteration 0: 0.692651
Cost after iteration 100: 0.666992
Cost after iteration 200: 0.573190
Cost after iteration 300: 0.390628
Cost after iteration 400: 0.262913
Cost after iteration 500: 0.195476
Cost after iteration 600: 0.157970
Cost after iteration 700: 0.135018
Cost after iteration 800: 0.119744
Cost after iteration 900: 0.108919
Cost after iteration 1000: 0.100879
Cost after iteration 1100: 0.094689
Cost after iteration 1200: 0.089789
Cost after iteration 1300: 0.085819
Cost after iteration 1400: 0.082539
Cost after iteration 1500: 0.079786
Cost after iteration 1600: 0.077440
Cost after iteration 1700: 0.075418
Cost after iteration 1800: 0.073654
Cost after iteration 1900: 0.072102
Cost after iteration 2000: 0.070724
Cost after iteration 2100: 0.069491
Cost after iteration 2200: 0.068381
Cost after iteration 2300: 0.067375
Cost after iteration 2400: 0.066459
Cost after iteration 2500: 0.065621
Cost after iteration 2600: 0.064851
Cost after iteration 2700: 0.064140
Cost

### Accuracy on the training set


In [22]:
# Print accuracy
predictions = predict(parameters, X)
print(accuracy_score(Y.T, predictions.T))

0.989010989010989


### Accuracy on the test set


In [23]:
predictions_test = predict(parameters, testx)
print(accuracy_score(testy.T, predictions_test.T))

0.9649122807017544


### On hidden layer size and learning rate

Larger hidden layers fit the training set more closely, up to the point where they start fitting its noise. The learning rate cuts both ways: too small and convergence takes far more iterations than budgeted, too large and the updates overshoot and the cost diverges.


## 4. Comparing activation functions

The hidden layer above uses `tanh`. Swapping it for `sigmoid` and for `ReLU`, holding everything else fixed, isolates the effect of that one choice.


The three are trained under identical conditions and compared on convergence and final accuracy.


In [27]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0) * 1.0

def sigmoid_derivative(a):
    return a * (1 - a)

def forward_propagation_bonus(X, parameters, activation):
    W1, b1, W2, b2 = parameters["W1"], parameters["b1"], parameters["W2"], parameters["b2"]
    
    Z1 = np.dot(W1, X) + b1
    
    if activation == "tanh":
        A1 = np.tanh(Z1)
    elif activation == "sigmoid":
        A1 = sigmoid(Z1)
    elif activation == "relu":
        A1 = relu(Z1)
        
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2) # always sigmoid for binary classification
    
    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

# Flexible Backward Propagation
def backprop_bonus(parameters, cache, X, Y, activation):
    m = X.shape[1]
    W2 = parameters["W2"]
    A1, A2, Z1 = cache["A1"], cache["A2"], cache["Z1"]
    
    dZ2 = A2 - Y
    dW2 = (1/m) * np.dot(dZ2, A1.T)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)
    
    # Calculating dZ1 based on the activation function derivative
    if activation == "tanh":
        dZ1 = np.dot(W2.T, dZ2) * (1 - np.power(A1, 2))
    elif activation == "sigmoid":
        dZ1 = np.dot(W2.T, dZ2) * sigmoid_derivative(A1)
    elif activation == "relu":
        dZ1 = np.dot(W2.T, dZ2) * relu_derivative(Z1)
        
    dW1 = (1/m) * np.dot(dZ1, X.T)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)
    
    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads

# Flexible Model Function
def NeuralNetwork_Bonus(X, Y, n_h, activation="tanh", num_iterations=10000, learning_rate=0.01, print_cost=False):
    np.random.seed(3)
    n_x = X.shape[0]
    n_y = Y.shape[0]
    parameters = initialize_parameters(n_x, n_h, n_y)
    
    for i in range(0, num_iterations):
        A2, cache = forward_propagation_bonus(X, parameters, activation)
        cost = compute_cost(A2, Y)
        grads = backprop_bonus(parameters, cache, X, Y, activation)
        parameters = update(parameters, grads, learning_rate)
        
        if print_cost and i % 5000 == 0:
            print (f"{activation.capitalize()} Cost after iteration {i}: {cost}")
            
    return parameters

# the experiments

activations_to_test = ["tanh", "sigmoid", "relu"]
results = {}

print("=== Training Models with Different Activations ===")

for act in activations_to_test:
    
    # Train
    print(f"\nTraining with {act}...")
    params_bonus = NeuralNetwork_Bonus(X, Y, n_h=10, activation=act, num_iterations=10000, learning_rate=0.01, print_cost=True)
    
    # Predict
    A2_train, _ = forward_propagation_bonus(X, params_bonus, act)
    pred_train = (A2_train > 0.5)
    acc_train = accuracy_score(Y.T, pred_train.T)
    
    A2_test, _ = forward_propagation_bonus(testx, params_bonus, act)
    pred_test = (A2_test > 0.5)
    acc_test = accuracy_score(testy.T, pred_test.T)
    
    results[act] = {"Train Accuracy": acc_train, "Test Accuracy": acc_test}

print("\n=== Final Results ===")
for act, res in results.items():
    print(f"Activation: {act.ljust(10)} | Train Acc: {res['Train Accuracy']:.4f} | Test Acc: {res['Test Accuracy']:.4f}")

=== Training Models with Different Activations ===

Training with tanh...
Tanh Cost after iteration 0: 0.6926509357420845
Tanh Cost after iteration 5000: 0.055514456431582135

Training with sigmoid...
Sigmoid Cost after iteration 0: 0.6932828224415243
Sigmoid Cost after iteration 5000: 0.12905151885518906

Training with relu...
Relu Cost after iteration 0: 0.6929283835585885
Relu Cost after iteration 5000: 0.05518324749344035

=== Final Results ===
Activation: tanh       | Train Acc: 0.9890 | Test Acc: 0.9649
Activation: sigmoid    | Train Acc: 0.9846 | Test Acc: 0.9649
Activation: relu       | Train Acc: 0.9890 | Test Acc: 0.9737


In [28]:
print("Based on the experiment results, significant differences in convergence speed and generalization were observed between the activation functions. Both ReLU and Tanh converged much faster than Sigmoid, achieving a significantly lower final cost (~0.055 vs. ~0.129) after the same number of iterations. This highlights the vanishing gradient problem associated with Sigmoid, where small gradients in the saturation regions slow down learning. While Tanh and ReLU achieved identical high training accuracy (98.90%), ReLU demonstrated superior generalization, achieving the highest test accuracy of 97.37% compared to 96.49% for the other two. Therefore, for this specific dataset, ReLU proved to be the most effective choice, offering both rapid convergence and the best performance on unseen data.")

Based on the experiment results, significant differences in convergence speed and generalization were observed between the activation functions. Both ReLU and Tanh converged much faster than Sigmoid, achieving a significantly lower final cost (~0.055 vs. ~0.129) after the same number of iterations. This highlights the vanishing gradient problem associated with Sigmoid, where small gradients in the saturation regions slow down learning. While Tanh and ReLU achieved identical high training accuracy (98.90%), ReLU demonstrated superior generalization, achieving the highest test accuracy of 97.37% compared to 96.49% for the other two. Therefore, for this specific dataset, ReLU proved to be the most effective choice, offering both rapid convergence and the best performance on unseen data.


## B. Neural network: multiclass classification

The same architecture extended to multiple classes with a softmax output layer, applied to the Vehicle Silhouettes dataset (`Vehicles.csv`): a 70/15/15 split with seed 777, reporting per-class metrics and a confusion matrix alongside accuracy.

Implemented in `vehicle-silhouettes-multiclass.ipynb` in this folder.

Dataset details: https://archive.ics.uci.edu/ml/datasets/Statlog+%28Vehicle+Silhouettes%29


## C. Neural network: regression

The same architecture adapted to regression, with a linear output unit and a mean-squared-error cost in place of cross-entropy, applied to the NASA Airfoil Self-Noise dataset (`Airfoil.csv`): a 70/15/15 split with seed 777, reported with MSE and MAE.

Implemented in `airfoil-noise-regression.ipynb` in this folder.

Dataset details: https://archive.ics.uci.edu/ml/datasets/Airfoil+Self-Noise


## References

- [Neural Networks and Deep Learning](https://www.coursera.org/learn/neural-networks-deep-learning)
- [Neural Network Visualization](http://scs.ryerson.ca/~aharley/neural-networks/)
- [CS231n: Neural Networks Case Study](http://cs231n.github.io/neural-networks-case-study/)
